# Chapter 15. Transformers for Natural Language Processing and Chatbots

Transformers became very good an language processing tasks, specifically translation due in part to their ability to easily scale across GPUs. They do not feature convolutional or _ layers.

For translation the 'original' transformer was the Encoder-Decoder architecture which followed these steps:

1. The encoder takes in some source text, and generates 'contextual embeddings' (what are these based on?)
2. The embeddings (one per token) are fed to the decoder alongside the output so-far
3. The decoder predicts the next output token for each input token
4. The last token output by the decoder is appended to the final output
5. Steps 2-4 are repeated until an end-of-sequence token is generated.

So intuitively the *encoder* is like the _reader_ which can see tokens bi-directionally, this is helpful for generating the contextual embeddings. It follows that the *decoder* is the _writer_ as it cannot see the next token(s) (masked) and solely generates text left to right as one might read or write a book in English (where we typically read left to right).

> _Contextual Embeddings_ are vectors that are cacluated using the Self-Attention mechanism which is a means of mathematically representing the relationship between tokens within an input i.e. the word 'bank' in 'The bank of the river'.

Eventually the big LLMs that we use most often (ChatGPT, Claude, Qwen, Deepseek, Kimi, Mistral, etc.) moved to a _decoder only_ architecture, since text generation is all that is really needed, and these models scale much better with billions (or trillions) of parameters and massive pre-training datasets. 

## Attention is all you need


> “Apart from the embedding layer, every layer in the encoder takes as input a tensor of shape [batch size, max English sequence length in the batch, embedding size] and returns a tensor of the exact same shape. This means that token representations get gradually transformed, hence the name of the architecture.”

The original architecture from the paper involves embeddings for inputs and outputs for the encoder and decoder respectively, of note:
- skip connections to prevent gradient vanishing
- multi-headed attention (self-attention) mechanisms put tokens in context with all others
    - in the decoder this is masked such that it only sees the previous tokens
- positional encodings to give the model an understanding of where a token is within the input

### Positional encodings
A simple _Embedding_ layer can represent the dense vector that encodes the position to the ith token's embedding in each sentence.

In [2]:
import torch
import torch.nn.functional as F
from torch import nn


class PosistionalEmbedding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        return self.dropout(X + self.pos_embed[: X.size(1)])


These days there are other encodings you might train including the popular _rotary positional encoding_ (RoPE) method.

### Multi-Head Attention

With the position encoded the meaning can be gleaned from a token by the MHA mechanism. This uses a dot product taking in *Q*, *K*, and *V* as inputs, where Q is a matrix representing a _query_, K is a matrix representing a key, and V is a matrix representing a value. Multiple are used at once to give the model a complete picture of the tokens 'meaning' by it's semantic meaning in the vocabulary, to the position that its in and the surrounding tokens. 

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1, 2)

    def forward(self, query, key, value):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        scores = q @ k.transpose(2, 3) / self.d**0.5
        weights = scores.softmax(dim=-1)
        Z = self.dropout(weights)
        Z = Z.transpose(1, 2)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)
